In [2]:
import numpy as np
import pandas as pd
import torch
import einops as eops

In [3]:
import torch.nn as nn 
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

In [4]:
class HospitalSelectorModel(nn.Module):
    def __init__(self, _nFeature :int):
        super(HospitalSelectorModel, self).__init__()

        self._nFeature = _nFeature
        self.model =  nn.Sequential(
            nn.Linear(_nFeature, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
        
    def forward(self, inp :np.ndarray):
        return self.model(inp)

   

In [52]:
class Train():
    def __init__(self, _nFeature :int):
        self.model = HospitalSelectorModel(_nFeature)
        self.loss = nn.BCEWithLogitsLoss()
        self.optim = torch.optim.Adam(self.model.parameters(), lr=0.01)

    def train(self, X :np.ndarray, Y :np.ndarray, epochs:int=100, _nbatch:int=32):
        print("Training starts")
        model = self.model
        optim = self.optim
        loss_fn = self.loss

        Y = np.expand_dims(Y, -1)
        X = torch.from_numpy(X).float()
        Y = torch.from_numpy(Y).float()

        inp = TensorDataset(X, Y)
        print(type(inp), inp)
        data_loader = DataLoader(inp, batch_size=_nbatch, shuffle=True)
            
        for epoch in range(epochs):
            model.train()

            for i, (train_X, train_Y) in enumerate(data_loader):
                optim.zero_grad()
                logits = model(train_X)
                logit_loss = loss_fn(logits, train_Y)
                logit_loss.backward()
                optim.step()

            with torch.no_grad():
                idx = epoch%X.shape[0]
                pred = model(X[idx:idx+1])
                loss = loss_fn(pred, Y[idx:idx+1])
                print(f"Finishing epoch: {epoch} with {loss.item()*100}% logit loss")


    #assume inp: n-hospital with m-features each hospital -> min(top k-pred, #pred>=50%)
    def pred(self, inp :np.ndarray, k :int=3):
        with torch.no_grad():
            inp = torch.Tensor(inp)
            pred_score = torch.sigmoid(self.model(inp))
            idx = []
            print(pred_score)
            for pred in pred_score:
                arg = torch.argmax(pred_score)
                if pred_score[arg]<0.8:
                    break
                pred_score[arg] = 0
                idx.append(arg)
                
            l = len(idx)
            return idx[:min(l, k)] 

    def save(self, name :str):
        torch.save(self.model.state_dict(), name)

In [43]:
data = pd.read_csv("Data.csv")

data


,Patient_ID,Distance_km,Budget_USD,Time_Mins,Criticality_Level,ICU_Beds,Heart_Issue,Cardiology_Dept,Brain_Issue,Neurology_Dept,Final_Score,Final_Recommendation
0,P001,15,5000,45,Medium,20,1,1,0,1,85,Go
1,P002,25,3000,60,Low,5,1,0,0,0,15,No
2,P003,5,8000,15,High,50,0,1,1,1,105,Go
3,P004,10,4500,30,Medium,12,0,0,1,0,25,No
4,P005,40,2000,90,Low,0,1,1,1,1,75,Go
...,...,...,...,...,...,...,...,...,...,...,...,...
395,P396,3,9800,12,Critical,38,1,0,1,0,13,No
396,P397,13,5900,37,Medium,16,1,1,1,0,39,No
397,P398,28,3600,70,Low,7,1,0,1,1,32,No
398,P399,9,6800,22,High,32,0,1,0,1,78,Go


In [44]:
x,y

(array([[-0.1683702455238285, -0.1316348584468005, 0.072946667353666, ...,
         0, 0, 1],
        [0.7417391897401087, -1.0092005814254705, 0.6982038160993747, ...,
         0, 1, 0],
        [-1.0784796807877657, 1.1847137260212044, -1.1775676301377513,
         ..., 1, 0, 0],
        ...,
        [1.0147720203192898, -0.7459308645318695, 1.1150419152631803, ...,
         0, 1, 0],
        [-0.7144359066821908, 0.6581742922340025, -0.8857809607230872,
         ..., 1, 0, 0],
        [0.4687063591609275, -0.526539433787202, 0.48978476651747177, ...,
         0, 0, 1]], shape=(400, 12), dtype=object),
 array([0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1,
        0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1,
        0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1,
        1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1,
        0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1,
        0, 1, 0, 1

In [45]:

y = data['Final_Recommendation'] = data['Final_Recommendation'].astype('category').cat.codes
x = data.drop(['Final_Recommendation', 'Patient_ID', 'Final_Score'], axis=1)


In [46]:
continous_cols = ['Distance_km', 'Budget_USD', 'Time_Mins', 'ICU_Beds']

x[continous_cols] = (x[continous_cols] - x[continous_cols].mean())/x[continous_cols].std()

In [47]:
x['Criticality_Level'] = data['Criticality_Level'].astype('category').cat.codes
x = pd.get_dummies(x, columns=['Criticality_Level'], dtype=int)

In [49]:
x = x.to_numpy()
y = y.to_numpy()

In [53]:
model = Train(12)
model.train(x, y, _nbatch=32, epochs=224)

Training starts
<class 'torch.utils.data.dataset.TensorDataset'> <torch.utils.data.dataset.TensorDataset object at 0x00000224BDF99090>
Finishing epoch: 0 with 75.17586946487427% logit loss
Finishing epoch: 1 with 15.430115163326263% logit loss
Finishing epoch: 2 with 0.0812530517578125% logit loss
Finishing epoch: 3 with 0.04375616554170847% logit loss
Finishing epoch: 4 with 1.7493247985839844% logit loss
Finishing epoch: 5 with 0.0025864361305139028% logit loss
Finishing epoch: 6 with 0.23764187935739756% logit loss
Finishing epoch: 7 with 1.7356567084789276% logit loss
Finishing epoch: 8 with 0.00476837158203125% logit loss
Finishing epoch: 9 with 0.17373176524415612% logit loss
Finishing epoch: 10 with 0.40645599365234375% logit loss
Finishing epoch: 11 with 0.06991777336224914% logit loss
Finishing epoch: 12 with 0.004482269287109375% logit loss
Finishing epoch: 13 with 0.00029510729291359894% logit loss
Finishing epoch: 14 with 0.36025047302246094% logit loss
Finishing epoch: 15 

In [54]:
preds = model.pred(x[0:20, :], 20)
for pred in preds:
    print(True if y[pred.item()]==1 else False)

tensor([[8.8395e-06],
        [1.0000e+00],
        [9.1327e-08],
        [1.0000e+00],
        [5.1625e-06],
        [1.0000e+00],
        [1.0000e+00],
        [9.9998e-01],
        [2.2001e-08],
        [9.9999e-01],
        [1.9627e-05],
        [1.0000e+00],
        [1.9464e-07],
        [1.0000e+00],
        [1.7772e-05],
        [1.0000e+00],
        [9.9999e-01],
        [1.0000e+00],
        [3.6612e-08],
        [9.9999e-01]])
True
True
True
True
True
True
True
True
True
True
True
True


In [21]:
y[0:20]

array([1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1])

In [55]:
model.save("HospSelectorModel.pth")